# Calculate Scene NDVI  

In this notebook, I develop a function that takes sentinel-2 observations, a given geometry (e.g. Phoenix Park boundary) in EPSG:32629 crs, and returns a mean_ndvi score

In [ ]:
# pseudocode for function:

def calculate_scene_ndvi(item, park_utm):
    # load three bands (NIR, Red, SCL) from STAC item
    # crop to park bounding box
    # mask to park geometry
    # keep vegetation pixels
    # calculate NDVI per pixel
    # calculate mean NDVI
    # return mean NDVI

Fill in the function using code from notebook 1:

In [1]:
# Load packages
import geopandas as gpd
from odc.stac import load
import rioxarray
import numpy as np
import pystac_client
import planetary_computer


In [6]:
def calculate_scene_ndvi(item, park_utm):
    # sign item:
    item = planetary_computer.sign(item)
    # load three bands (NIR, Red, SCL) from STAC item
    ds = load(
        [item],
        bands=["B04", "B08", "SCL"],
        crs="EPSG:32629",
        resolution=10,
    )

    # crop to park bounding box
    xmin, ymin, xmax, ymax = park_utm.total_bounds
    park_bbox = ds.sel(
        x = slice(xmin, xmax),
        y = slice(ymax, ymin)
    )

    # mask to park geometry
    park_bbox = park_bbox.rio.write_crs("EPSG:32629")
    park_masked = park_bbox.rio.clip(
        park_utm.geometry,
        park_utm.crs, 
        drop=False 
    )

    # keep vegetation pixels
    vegetation_mask = park_masked.SCL==4

    # calculate NDVI per pixel
    ndvi = (
        (park_masked.B08 - park_masked.B04)
        / (park_masked.B08 + park_masked.B04)
    )
    ndvi_veg = ndvi.where(vegetation_mask)
    #for now just return ndvi_veg
    return ndvi_veg

    # calculate mean NDVI
    # return mean NDVI

Test the function thus far

In [7]:
# Get park_utm
parks = gpd.read_file('../data/dcc_parks_strategy2016_park_classification.geojson')
park = parks[parks.Name.str.contains('Phoenix')].copy()
park_utm = park.to_crs('EPSG:32629')

In [8]:
# Get item from STAC catalogue
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1'
    )

search = catalog.search(
    collections = ['sentinel-2-l2a'],
    bbox = park.total_bounds,
    datetime='2026-07-01/2026-07-31'
)

items = list(search.items())

item = items[13]

In [9]:
test_ndvi = calculate_scene_ndvi(item, park_utm)

In [12]:
test_ndvi.mean().item()

0.5580050945281982

Same result as in notebook 1, so the function works so far!  

Now let's add the final steps to return the mean ndvi from the funtion

In [13]:
def calculate_scene_ndvi(item, park_utm):
    # sign item:
    item = planetary_computer.sign(item)
    # load three bands (NIR, Red, SCL) from STAC item
    ds = load(
        [item],
        bands=["B04", "B08", "SCL"],
        crs="EPSG:32629",
        resolution=10,
    )

    # crop to park bounding box
    xmin, ymin, xmax, ymax = park_utm.total_bounds
    park_bbox = ds.sel(
        x = slice(xmin, xmax),
        y = slice(ymax, ymin)
    )

    # mask to park geometry
    park_bbox = park_bbox.rio.write_crs("EPSG:32629")
    park_masked = park_bbox.rio.clip(
        park_utm.geometry,
        park_utm.crs, 
        drop=False 
    )

    # keep vegetation pixels
    vegetation_mask = park_masked.SCL==4

    # calculate NDVI per pixel
    ndvi = (
        (park_masked.B08 - park_masked.B04)
        / (park_masked.B08 + park_masked.B04)
    )
    ndvi_veg = ndvi.where(vegetation_mask)

    # calculate mean NDVI
    ndvi_mean = ndvi_veg.mean().item()

    # return mean NDVI
    return ndvi_mean

In [14]:
test_ndvi = calculate_scene_ndvi(item, park_utm)
test_ndvi

0.5580050945281982